# 🚪 controller_explore — the bouncer's judgment (M4.1 →)

Everything the controller ever decides is in this notebook's two objects: a **pure
function** (the predicate — six checks, first failure wins) and a **dict** (the session
state machine). No LLM, no I/O, no side effects — you can hold the entire authorization
logic of the system in your head, and this notebook exists so you literally do.

Grows with Phase 4: §1–§3 predicate + machine (M4.1) · auth (M4.2) · translators (M4.3) ·
the HTTP API (M4.4) land as new sections.

**Companions:** [`docs/05-controller-spec.md`](../../docs/05-controller-spec.md) (the
rulebook) · `controller/src/controller/domain.py` (≈100 lines, half comments) ·
`controller/tests/test_domain.py` (every cell here, pinned).

## 1. The predicate: six boring checks, in a contract order

`CANONICAL_ENTITLEMENT_VIEW` is ticket #7 exactly as the controller reads it off the
chain. At 14:02, with Ada asking, everything passes:

In [1]:
from a2a_interfaces import ErrorCode
from a2a_interfaces.fixtures import ADA, BELL, CANONICAL_ENTITLEMENT_VIEW, WINDOW
from controller.domain import predicate, transition, IllegalTransition, TRANSITIONS

VIEW = CANONICAL_ENTITLEMENT_VIEW
verdict = predicate(VIEW, owner=ADA, requester=ADA, now=WINDOW.start + 120, active_ids=set())
print("14:02, Ada asks →", verdict, "  (None = come in)")


14:02, Ada asks → None   (None = come in)


## 2. Flip one fact at a time, watch the named refusal

Each denial is an `ErrorCode` from the shared enum (docs/03 §3.4) — the same strings the
HTTP API will return (M4.4) and the agents will read (M5.x):

In [2]:
IN_WINDOW = WINDOW.start + 120

cases = {
    "Bell asks instead of Ada":  dict(view=VIEW, owner=ADA, requester=BELL, now=IN_WINDOW),
    "Ada asks at 13:00":         dict(view=VIEW, owner=ADA, requester=ADA, now=WINDOW.start - 3600),
    "Ada asks at 16:00 sharp":   dict(view=VIEW, owner=ADA, requester=ADA, now=WINDOW.end),
    "ticket was revoked":        dict(view=VIEW.model_copy(update={"revoked": True}), owner=ADA, requester=ADA, now=IN_WINDOW),
    "serviceType nobody knows":  dict(view=VIEW.model_copy(update={"service_type": 7}), owner=ADA, requester=ADA, now=IN_WINDOW),
}
for story, kw in cases.items():
    print(f"{story:28} → {predicate(active_ids=set(), **kw)}")

print(f"{'#7 already has a session':28} → {predicate(VIEW, ADA, ADA, IN_WINDOW, active_ids={7})}")


Bell asks instead of Ada     → ErrorCode.E_NOT_OWNER
Ada asks at 13:00            → ErrorCode.E_NOT_STARTED
Ada asks at 16:00 sharp      → ErrorCode.E_EXPIRED
ticket was revoked           → ErrorCode.E_REVOKED
serviceType nobody knows     → ErrorCode.E_SCOPE
#7 already has a session     → ErrorCode.E_CONFLICT


**The order is contract, not accident.** A request that fails *everything* still
answers `E_NOT_OWNER` — first check, first answer. Try to predict, then run:

In [3]:
wreck = VIEW.model_copy(update={"revoked": True, "service_type": 9})
print(predicate(wreck, owner=ADA, requester=BELL, now=WINDOW.end + 999, active_ids={7}))


ErrorCode.E_NOT_OWNER


## 3. The state machine is a dict you can read

Not `if`-chains — data. Legal moves are keys; terminal states absorb `teardown`
(rule 8: tearing down what's already down is a success); everything else raises
`IllegalTransition`, which means a *programming* error, never a user denial:

In [4]:
from a2a_interfaces import SessionState

for (state, event), nxt in TRANSITIONS.items():
    print(f"{state.value:11} --{event:17}→ {nxt.value}")

print()
print("torn_down --teardown→", transition(SessionState.TORN_DOWN, "teardown").value, " (absorbed)")
try:
    transition(SessionState.REQUESTED, "provision_ok")   # skipped authorization!
except IllegalTransition as err:
    print("requested --provision_ok→ raises:", err)


requested   --authorize        → authorized
requested   --deny             → failed
authorized  --provision_ok     → active
authorized  --provision_failed → failed
active      --teardown         → torn_down

torn_down --teardown→ torn_down  (absorbed)
requested --provision_ok→ raises: no transition for event 'provision_ok' in state 'requested'


## 4. Rule 4, checked with your own eyes

The domain imports nothing that can do I/O — run the same inspection the test suite
pins (`test_domain_purity_no_io_imports`):

In [5]:
import ast, inspect
from controller import domain

tree = ast.parse(inspect.getsource(domain))
imports = sorted({
    (a.name if isinstance(n, ast.Import) else n.module).split(".")[0]
    for n in ast.walk(tree)
    if isinstance(n, (ast.Import, ast.ImportFrom))
    for a in (n.names if isinstance(n, ast.Import) else [n])
    if (a.name if isinstance(n, ast.Import) else n.module)
})
print("domain.py imports:", imports)
assert set(imports) <= {"__future__", "a2a_interfaces"}
print("no web3, no pygnmi, no HTTP, no filesystem — the judgment is portable ✓")


domain.py imports: ['__future__', 'a2a_interfaces']
no web3, no pygnmi, no HTTP, no filesystem — the judgment is portable ✓


## 5. The auth dance (M4.2): prove you hold the key, show nobody the key

Owning ticket #7 is on-chain fact; being *the one asking right now* is not. The
challenge–response closes that gap: the controller issues a single-use nonce, the owner
signs the docs/03 §3.2 string (plain EIP-191 — no chain transaction!), the controller
recovers the signer and compares with `ownerOf`. Watch every way it can fail:

In [6]:
from eth_account import Account
from eth_account.messages import encode_defunct
from controller.auth import AuthStore, proof_message

ada_key = Account.create("notebook-ada")       # a throwaway 'Ada' — keys stay in cells here
store = AuthStore("bw-ctrl-1")
NOW = WINDOW.start + 120

challenge = store.issue(7, now=NOW)
print("challenge:", challenge, "\n")

message = proof_message(store.controller_id, challenge.nonce, 7, challenge.expires_at)
print("she signs:", message)
signature = ada_key.sign_message(encode_defunct(text=message)).signature.hex()

print("\nverify (right key):   ", store.verify(7, challenge.nonce, signature, owner=ada_key.address, now=NOW + 5))
print("verify again (replay):", store.verify(7, challenge.nonce, signature, owner=ada_key.address, now=NOW + 6))

thief = Account.create("notebook-thief")
c2 = store.issue(7, now=NOW)
sig2 = thief.sign_message(encode_defunct(text=proof_message(store.controller_id, c2.nonce, 7, c2.expires_at))).signature.hex()
print("verify (thief's key): ", store.verify(7, c2.nonce, sig2, owner=ada_key.address, now=NOW))

challenge: Challenge(nonce='0x0ba5dfc33b81bdf85ed8278fab20cc1b', controller_id='bw-ctrl-1', expires_at=1757945220) 

she signs: a2a-activate|bw-ctrl-1|0x0ba5dfc33b81bdf85ed8278fab20cc1b|7|1757945220

verify (right key):    None
verify again (replay): ErrorCode.E_NONCE_REUSED
verify (thief's key):  ErrorCode.E_NOT_OWNER


## 6. Translators (M4.3): terms → the exact calls, and nothing else

The ticket says *what* (50 Mbps, path `0x…07`); the provisioner needs *where and how*
(`srl1`, `ethernet-1/1`). The translator joins them through `resource_map.yaml` — the
ONLY file in the system where on-chain handles meet device names (ADR-005). Output is
data, not action: a list of intended calls the wiring will apply.

In [7]:
from a2a_interfaces.fixtures import TELEMETRY_ENTITLEMENT_VIEW
from controller.resource_map import load_resource_map
from controller.translators import translate, UnmappedResource

resource_map = load_resource_map()
print("the map knows:", [("0x…" + k.hex()[-2:], type(v).__name__) for k, v in resource_map.items()], "\n")

for view in (VIEW, TELEMETRY_ENTITLEMENT_VIEW):
    for call in translate(f"ent{view.id}-a1", view, resource_map):
        print(f"ticket #{view.id} → {call.method}({', '.join(f'{k}={v}' for k, v in call.kwargs.items())})\n")

stranger = VIEW.model_copy(update={"resource_id": b"\x99" * 32})
try:
    translate("s", stranger, resource_map)
except UnmappedResource as err:
    print("unknown resource →", type(err).__name__, err, " (the wiring answers E_SCOPE)")

the map knows: [('0x…07', 'ResolvedPath'), ('0x…08', 'ResolvedNode')] 

ticket #7 → apply_bandwidth(session_id=ent7-a1, path=device='srl1' ingress_if='ethernet-1/1' egress_if='ethernet-1/2', capacity_bps=50000000, qos_class=1)

ticket #8 → apply_telemetry(session_id=ent8-a1, target=device='srl1', sensor_paths=['/interface[name=ethernet-1/1]/statistics'], collector_endpoint=10.0.0.50:57000, sample_interval_s=10)

unknown resource → UnmappedResource 0x9999999999999999999999999999999999999999999999999999999999999999  (the wiring answers E_SCOPE)


## 7. The HTTP door (M4.4): the whole §3 API, in-process

`build_app(service)` is FastAPI with zero logic — parse, call, map codes to statuses.
Because the service runs on injected ports, you can stand the ENTIRE controller up on
cardboard right here in a cell and drive it with HTTP requests:

In [8]:
from fastapi.testclient import TestClient
from a2a_interfaces.fixtures import BANDWIDTH_NEED, TICKET_ID
from e2e.skeleton.fakes import FakeChain, FakeClock
from e2e.skeleton.scripted_agents import ScriptedProvider
from netctl.mock import MockProvisioner
from controller.app import build_app
from controller.auth import AuthStore, proof_message
from controller.resource_map import load_resource_map
from controller.service import ControllerService

nb_ada = Account.create("nb-http-ada")
clock = FakeClock(WINDOW.start - 60)
chain = FakeChain(clock, balances={nb_ada.address: 10**20}, next_id=TICKET_ID)
chain.fulfill(ScriptedProvider().quote(BANDWIDTH_NEED), buyer=nb_ada.address)
clock.advance(180)  # inside the window

service = ControllerService(chain, MockProvisioner(), AuthStore("bw-ctrl-1"), load_resource_map())
client = TestClient(build_app(service))

ch = client.post("/v0/challenge", json={"entitlement_id": 7}).json()
print("challenge  →", ch)
msg = proof_message(ch["controller_id"], ch["nonce"], 7, ch["expires_at"])
sig = "0x" + nb_ada.sign_message(encode_defunct(text=msg)).signature.hex()
act = client.post("/v0/activate", json={"entitlement_id": 7, "action": {"kind": "bandwidth"},
                                        "proof": {"nonce": ch["nonce"], "signature": sig}})
print("activate   →", act.status_code, act.json())
replay = client.post("/v0/activate", json={"entitlement_id": 7, "action": {"kind": "bandwidth"},
                                           "proof": {"nonce": ch["nonce"], "signature": sig}})
print("replay     →", replay.status_code, replay.json())
print("teardown   →", client.post("/v0/teardown", json={"session_id": act.json()["session_id"]}).json())

challenge  → {'v': 0, 'nonce': '0x31aa97b925c0484b0a9456e495ba7fce', 'controller_id': 'bw-ctrl-1', 'expires_at': 1757945220}
activate   → 200 {'v': 0, 'session_id': 'ent7-a0', 'entitlement_id': 7, 'state': 'active', 'since': 1757944920, 'expires_at': 1757952000}
replay     → 401 {'error': 'E_NONCE_REUSED'}
teardown   → {'state': 'torn_down'}


/home/musel/Github/a2a-tokenized-provisioning/.venv/lib/python3.13/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## Experiments to try

- The window check: find the exact second Ada's ticket dies (hint: §2's third case).
  Why `now >= end_time` and not `>`? What would one extra second cost the provider?
- Give the predicate `known_service_types=(0,)` and a telemetry view — this is exactly
  how the M0.3 stub scopes itself to what it can translate.
- Draw `TRANSITIONS` as a diagram from the dict alone; compare with docs/05 §3.
- Break it: in `domain.py`, swap the order of the revoked and expired checks, run
  `uv run pytest controller/` — which named test catches it?